#  Module 2: Data Engineering -- Silver & Gold Layers

## Overview

This notebook implements the **Medallion Architecture** (Bronze -> Silver -> Gold) for healthcare data:

```
+--------------+    +--------------+    +--------------+
|    BRONZE    |--->|    SILVER    |--->|     GOLD     |
|  Raw CSV     |    |  Cleansed &  |    |  Business    |
|  as-ingested |    |  Standardized|    |  KPIs &      |
|              |    |              |    |  Analytics   |
+--------------+    +--------------+    +--------------+
```

### What You'll Build

**Part A -- Silver Layer** (cleansed, typed, standardized):
| Silver Table | Source | Key Transformations |
|---|---|---|
| `silver_patients` | bronze_patients | Age groups, risk categories |
| `silver_encounters` | bronze_encounters | Date parsing, LOS categories, temporal dimensions |
| `silver_conditions` | bronze_conditions | ICD-10 -> clinical category mapping |
| `silver_claims` | bronze_claims | Payment ratio, denial flag |
| `silver_medications` | bronze_medications | Date parsing |
| `silver_vitals` | bronze_vitals | Numeric casting, SIRS sepsis flag |
| `silver_clinical_notes` | bronze_clinical_notes | Date parsing |

**Part B -- Gold Layer** (business-ready KPIs):
| Gold Table | Healthcare Metric | Why It Matters |
|---|---|---|
| `gold_readmissions` | 30-day readmission rate | CMS penalizes up to 3% of Medicare payments |
| `gold_ed_utilization` | ED frequent flyers | 5% of patients drive 30% of ED costs |
| `gold_alos` | Avg length of stay by diagnosis | Each extra day costs $2,000-$3,000 |
| `gold_encounter_summary` | Volume with demographics | Demand forecasting & capacity planning |
| `gold_financial` | Revenue, denials, collections | $262B lost to denied claims annually |
| `gold_population_health` | Chronic disease prevalence | Proactive care reduces acute events |

### How to Use This Notebook

1. **Import** this notebook into your Fabric workspace
2. **Attach** your `HealthcareLakehouse` (Explorer pane -> Add data items -> From OneLake catalog -> select the **Lakehouse**, not the SQL Analytics Endpoint)
3. Click **> Run all** -- the entire notebook runs end-to-end in ~5 minutes

### PySpark Pattern Used Throughout

Every table follows the same 4-step pattern:
```python
df = spark.table("bronze_xxx")            # Step 1: Read from Bronze/Silver
silver = df.withColumn(...)                # Step 2: Transform columns
silver.write.mode("overwrite")             # Step 3: Write as Delta table
      .format("delta").saveAsTable(...)     
silver.groupBy(...).count().show()          # Step 4: Validate output
```

> [!] **Session Note:** If your Spark session expires, re-run all cells from the top using **Run all**. Fabric does not preserve variables across session restarts.

---
# Part A: Silver Layer -- Cleansed & Standardized Data

The Silver layer takes raw Bronze data and applies:
- **Type casting** -- Convert strings to proper dates, integers, and doubles (Bronze CSVs store everything as strings)
- **Computed columns** -- Derived fields that don't exist in the source but are essential for analytics
- **Standardization** -- Consistent categorization, null handling, and naming conventions

### Why This Matters in Healthcare
Healthcare data is notoriously messy. EHR systems export dates in different formats, numeric fields arrive as strings, and critical clinical categories (like "is this patient high-risk?") must be computed from raw data. The Silver layer creates a **single source of truth** that all downstream analytics can rely on.

## Setup: Import PySpark Libraries

We import all PySpark SQL functions and types we'll use throughout:
- `to_date()` / `to_timestamp()` -- Convert strings to proper Date/Timestamp types
- `col()` -- Reference a DataFrame column by name
- `when()` -- SQL `CASE WHEN` equivalent for conditional logic
- `round()` -- Round numeric values
- `date_format()`, `year()`, `quarter()`, `dayofweek()` -- Extract date components
- `IntegerType()` / `DoubleType()` -- Numeric type targets for casting

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

## Silver Patients

### Business Context
Patient demographics are the **foundation** of healthcare analytics. Every quality metric, financial report, and population health dashboard starts with "which patients are we talking about?"

### What We're Adding
| New Column | Logic | Why It Matters |
|---|---|---|
| `age_group` | Bucket age into 18-29, 30-44, 45-59, 60-74, 75+ | CMS requires reporting by age bracket. Population health programs target interventions by age. |
| `risk_category` | Low (<1.5), Medium (1.5-3.0), High (>=3.0) | Risk score determines care management intensity. High-risk patients (top 5%) drive 50% of spending. |

### Technical Notes
- `to_date()` converts string like `"1985-03-15"` to proper Date type (enables date arithmetic)
- `.cast(IntegerType())` converts string `"25"` to integer `25` (without this, `"25" > "3"` is False in string comparison!)
- `when().when().otherwise()` is PySpark's equivalent of SQL `CASE WHEN ... ELSE ... END`
- `.select()` explicitly lists output columns to control the schema

In [ ]:
# Read the raw Bronze patients table
patients = spark.table("bronze_patients")

# Build Silver layer with type casting and computed columns
silver_patients = patients \
    .withColumn("date_of_birth", to_date(col("date_of_birth"))) \
    .withColumn("age", col("age").cast(IntegerType())) \
    .withColumn("risk_score", col("risk_score").cast(DoubleType())) \
    .withColumn("age_group", 
        when(col("age") < 30, "18-29")
        .when(col("age") < 45, "30-44")
        .when(col("age") < 60, "45-59")
        .when(col("age") < 75, "60-74")
        .otherwise("75+")) \
    .withColumn("risk_category",
        when(col("risk_score") < 1.5, "Low")
        .when(col("risk_score") < 3.0, "Medium")
        .otherwise("High")) \
    .select(
        "patient_id", "first_name", "last_name", "date_of_birth", "age",
        "age_group", "gender", "race", "zip_code", "city", "state",
        "insurance_type", "primary_care_provider", "risk_score", "risk_category"
    )

# Write as Delta table (mode="overwrite" makes this safe to re-run)
silver_patients.write.mode("overwrite").format("delta").saveAsTable("silver_patients")
print(f"[x] silver_patients: {silver_patients.count()} rows")
silver_patients.show(5)

## Silver Encounters

### Business Context
Encounters are the **transaction records** of healthcare -- every patient interaction (ED visit, hospital admission, outpatient appointment) generates an encounter record. This is what we measure volume, cost, and quality against.

### What We're Adding
| New Column | Logic | Why It Matters |
|---|---|---|
| `encounter_month` | `"yyyy-MM"` format | Monthly volume trending, seasonality detection (flu season = Dec-Feb ED spike) |
| `encounter_year` / `encounter_quarter` | Extract year & quarter | Year-over-year comparisons, financial period alignment |
| `day_of_week` / `is_weekend` | 1=Sunday...7=Saturday | The **"weekend effect"** -- patients admitted on weekends have 10-15% higher mortality due to reduced specialist staffing |
| `los_category` | Same Day, Short (1-2), Medium (3-5), Long (6-10), Extended (>10) | Clinically meaningful buckets for operational analysis |

### Technical Notes
- `date_format(col, "yyyy-MM")` -> `"2024-03"` -- great for monthly grouping
- `dayofweek()` returns 1=Sunday through 7=Saturday in Spark
- LOS categories are clinically meaningful: Extended (>10 days) often signals ICU stays or discharge planning barriers

In [ ]:
encounters = spark.table("bronze_encounters")

silver_encounters = encounters \
    .withColumn("encounter_date", to_date(col("encounter_date"))) \
    .withColumn("discharge_date", to_date(col("discharge_date"))) \
    .withColumn("length_of_stay_days", col("length_of_stay_days").cast(IntegerType())) \
    .withColumn("total_charges", col("total_charges").cast(DoubleType())) \
    .withColumn("encounter_month", date_format(col("encounter_date"), "yyyy-MM")) \
    .withColumn("encounter_year", year(col("encounter_date"))) \
    .withColumn("encounter_quarter", quarter(col("encounter_date"))) \
    .withColumn("day_of_week", dayofweek(col("encounter_date"))) \
    .withColumn("is_weekend", 
        when(dayofweek(col("encounter_date")).isin(1, 7), True).otherwise(False)) \
    .withColumn("los_category",
        when(col("length_of_stay_days") == 0, "Same Day")
        .when(col("length_of_stay_days") <= 2, "Short (1-2 days)")
        .when(col("length_of_stay_days") <= 5, "Medium (3-5 days)")
        .when(col("length_of_stay_days") <= 10, "Long (6-10 days)")
        .otherwise("Extended (>10 days)"))

silver_encounters.write.mode("overwrite").format("delta").saveAsTable("silver_encounters")
print(f"[x] silver_encounters: {silver_encounters.count()} rows")
silver_encounters.groupBy("encounter_type").count().orderBy("count", ascending=False).show()

## Silver Conditions

### Business Context
Diagnoses are coded using **ICD-10-CM** (International Classification of Diseases) -- the universal language of clinical documentation. Raw codes like `"E11.9"` are meaningless to business users. This table maps them to **human-readable clinical categories**.

### ICD-10 Code Mapping
| ICD-10 Prefix | Category | National Prevalence | Clinical Significance |
|---|---|---|---|
| E11 | Diabetes | 37.3M Americans | Drives kidney failure, blindness, amputations |
| I50 | Heart Failure | 6.7M Americans | #1 cause of hospital readmissions |
| J44 | COPD | 16M Americans | 3rd leading cause of death |
| I10 | Hypertension | 119M Americans | "Silent killer" -- damages arteries over decades |
| E78 | Hyperlipidemia | 94M Americans | Primary driver of atherosclerosis |
| N18 | Chronic Kidney Disease | 37M Americans | Often comorbid with diabetes/HTN |
| F32 | Depression | 21M Americans | Doubles healthcare costs when comorbid |
| J45 | Asthma | 25M Americans | Leading chronic disease in children |
| I25 | Coronary Artery Disease | 20M Americans | #1 cause of death globally |
| E66 | Obesity | 100M+ Americans | Underpins diabetes, HTN, CAD, and more |

### Technical Notes
- We use `startswith("E11")` because ICD-10 codes are hierarchical -- `E11.0`, `E11.2`, `E11.9` are all diabetes subtypes
- Exception: Hypertension uses exact match `== "I10"` because I10 has no subtypes, and `startswith("I10")` would match other categories

In [ ]:
conditions = spark.table("bronze_conditions")

silver_conditions = conditions \
    .withColumn("date_diagnosed", to_date(col("date_diagnosed"))) \
    .withColumn("condition_category",
        when(col("condition_code").startswith("E11"), "Diabetes")
        .when(col("condition_code").startswith("I50"), "Heart Failure")
        .when(col("condition_code").startswith("J44"), "COPD")
        .when(col("condition_code") == "I10", "Hypertension")
        .when(col("condition_code").startswith("E78"), "Hyperlipidemia")
        .when(col("condition_code").startswith("N18"), "Chronic Kidney Disease")
        .when(col("condition_code").startswith("F32"), "Depression")
        .when(col("condition_code").startswith("J45"), "Asthma")
        .when(col("condition_code").startswith("I25"), "Coronary Artery Disease")
        .when(col("condition_code").startswith("E66"), "Obesity")
        .otherwise("Other"))

silver_conditions.write.mode("overwrite").format("delta").saveAsTable("silver_conditions")
print(f"[x] silver_conditions: {silver_conditions.count()} rows")
silver_conditions.groupBy("condition_category").count().orderBy("count", ascending=False).show()

## Silver Claims

### Business Context
Claims data is the **financial backbone** of healthcare analytics. Each claim = one bill submitted to an insurance payer. We add two critical computed fields:

| New Column | Logic | Why It Matters |
|---|---|---|
| `payment_ratio` | paid_amount / claim_amount | The single most important financial metric. 1.0 = paid in full, 0.83 = avg Medicare rate, 0.0 = denied |
| `is_denied` | True if claim_status = "Denied" | National avg denial rate: ~12%. Each denied claim costs $25-50 in administrative re-work |

### Technical Notes
- `when(col("claim_amount") > 0, ...)` guards against division by zero
- `round(..., 4)` gives 4 decimal places for ratio precision
- All monetary fields cast to `DoubleType()` for arithmetic operations

In [ ]:
claims = spark.table("bronze_claims")

silver_claims = claims \
    .withColumn("claim_date", to_date(col("claim_date"))) \
    .withColumn("claim_amount", col("claim_amount").cast(DoubleType())) \
    .withColumn("paid_amount", col("paid_amount").cast(DoubleType())) \
    .withColumn("denied_amount", col("denied_amount").cast(DoubleType())) \
    .withColumn("patient_responsibility", col("patient_responsibility").cast(DoubleType())) \
    .withColumn("days_to_payment", col("days_to_payment").cast(IntegerType())) \
    .withColumn("payment_ratio", 
        when(col("claim_amount") > 0, round(col("paid_amount") / col("claim_amount"), 4))
        .otherwise(0)) \
    .withColumn("is_denied", when(col("claim_status") == "Denied", True).otherwise(False))

silver_claims.write.mode("overwrite").format("delta").saveAsTable("silver_claims")
print(f"[x] silver_claims: {silver_claims.count()} rows")

## Silver Medications, Vitals & Clinical Notes

### Business Context

**Medications** -- Parsed start/end dates. In production, you'd also normalize drug names to RxNorm and check for drug-drug interactions.

**Vitals** -- The most important addition here is the **SIRS flag** (Systemic Inflammatory Response Syndrome), an early indicator of **sepsis** -- a life-threatening condition that kills 270,000 Americans per year.

SIRS criteria (simplified -- need all 3 in our model):
- Temperature > 100.4 degrees F OR < 96.8 degrees F (abnormal)
- Heart rate > 90 bpm (elevated)
- Respiratory rate > 20 breaths/min (elevated)

**Clinical Notes** -- Free-text physician notes with parsed timestamps. These will be processed by Azure OpenAI in a later module.

### Technical Notes
- `to_timestamp()` is used for vitals (not `to_date()`) because vitals need time-of-day precision
- PySpark boolean operators: `&` (AND), `|` (OR) -- each condition must be in parentheses for correct precedence

In [ ]:
# === Silver Medications ===
medications = spark.table("bronze_medications")
silver_medications = medications \
    .withColumn("start_date", to_date(col("start_date"))) \
    .withColumn("end_date", to_date(col("end_date")))

silver_medications.write.mode("overwrite").format("delta").saveAsTable("silver_medications")
print(f"[x] silver_medications: {silver_medications.count()} rows")

# === Silver Vitals (with SIRS sepsis early warning flag) ===
vitals = spark.table("bronze_vitals")
silver_vitals = vitals \
    .withColumn("timestamp", to_timestamp(col("timestamp"))) \
    .withColumn("heart_rate", col("heart_rate").cast(IntegerType())) \
    .withColumn("systolic_bp", col("systolic_bp").cast(IntegerType())) \
    .withColumn("diastolic_bp", col("diastolic_bp").cast(IntegerType())) \
    .withColumn("temperature_f", col("temperature_f").cast(DoubleType())) \
    .withColumn("respiratory_rate", col("respiratory_rate").cast(IntegerType())) \
    .withColumn("spo2_percent", col("spo2_percent").cast(IntegerType())) \
    .withColumn("pain_level", col("pain_level").cast(IntegerType())) \
    .withColumn("is_sirs_positive",
        (  (col("temperature_f") > 100.4) | (col("temperature_f") < 96.8) ) &
        (col("heart_rate") > 90) &
        (col("respiratory_rate") > 20)
    )

silver_vitals.write.mode("overwrite").format("delta").saveAsTable("silver_vitals")
print(f"[x] silver_vitals: {silver_vitals.count()} rows")

# === Silver Clinical Notes ===
clinical_notes = spark.table("bronze_clinical_notes")
silver_clinical_notes = clinical_notes \
    .withColumn("note_date", to_date(col("note_date")))

silver_clinical_notes.write.mode("overwrite").format("delta").saveAsTable("silver_clinical_notes")
print(f"[x] silver_clinical_notes: {silver_clinical_notes.count()} rows")

print("\n[OK] All Silver tables created!")

---
# Part B: Gold Layer -- Business-Ready Healthcare KPIs

The Gold layer contains **pre-computed, business-ready analytics tables** that Power BI dashboards and AI agents consume directly.

### Why Pre-Compute These Metrics?
- **Performance:** A 30-day readmission calculation requires a self-join across all encounters. Pre-computing it once means Power BI dashboards render in <1 second.
- **Consistency:** Every report uses the SAME definition of "readmission" or "frequent flyer" because the logic lives in one place.
- **Governance:** Business rule changes (e.g., CMS changes the readmission window) only need to be updated here.

## 30-Day Hospital Readmissions

### Business Context
A **30-day readmission** occurs when a patient is discharged from a hospital and returns for another inpatient admission within 30 days. This is one of the most important quality metrics in US healthcare:

- **CMS Hospital Readmissions Reduction Program (HRRP):** Hospitals with excess readmissions face Medicare payment reductions of up to **3%** of total payments
- **Patient impact:** A readmission signals the patient wasn't fully recovered or lacked adequate follow-up care
- **Financial impact:** Average inpatient stay costs $13,000 -- every preventable readmission = $13K in avoidable cost
- **National benchmark:** ~15% readmission rate. Above 15% triggers CMS scrutiny

### Technical Approach: Self-Join
We need to compare each encounter against ALL other encounters for the same patient. This requires a **self-join** -- joining the encounters table against itself:

```
Index Admission ("idx")  <-->  Potential Readmission ("readmit")
Same patient? [x]
Readmit after discharge? [x]  
Within 30 days? [x]
Not the same encounter? [x]
Patient didn't die or leave AMA? [x]
```

A `LEFT` join ensures all index admissions appear in output -- those without a readmission get `NULL` (which becomes `was_readmitted = False`).

In [ ]:
# Filter to Inpatient encounters only (CMS definition: readmissions only apply to inpatient stays)
encounters = spark.table("silver_encounters") \
    .filter(col("encounter_type") == "Inpatient")

# Create two aliases for the self-join
index_admissions = encounters.alias("idx")
readmissions = encounters.alias("readmit")

# Self-join: match each index admission with any readmission within 30 days
readmission_pairs = index_admissions.join(
    readmissions,
    (col("idx.patient_id") == col("readmit.patient_id")) &          # Same patient
    (col("readmit.encounter_date") > col("idx.discharge_date")) &    # Readmit AFTER discharge
    (datediff(col("readmit.encounter_date"), col("idx.discharge_date")) <= 30) &  # Within 30 days
    (col("idx.encounter_id") != col("readmit.encounter_id")) &       # Not the same encounter
    (col("idx.discharge_disposition") != "Expired") &                 # CMS exclusion: patient didn't die
    (col("idx.discharge_disposition") != "Against Medical Advice"),    # CMS exclusion: not AMA
    "left"  # Keep ALL index admissions, even those without readmission
)

# Build the output table with readmission flag
gold_readmissions = readmission_pairs.select(
    col("idx.encounter_id").alias("index_encounter_id"),
    col("idx.patient_id"),
    col("idx.encounter_date").alias("index_admission_date"),
    col("idx.discharge_date").alias("index_discharge_date"),
    col("idx.primary_diagnosis_code").alias("index_diagnosis_code"),
    col("idx.primary_diagnosis_description").alias("index_diagnosis"),
    col("idx.facility_name").alias("index_facility"),
    col("idx.attending_provider").alias("index_provider"),
    col("idx.length_of_stay_days").alias("index_los"),
    col("idx.discharge_disposition").alias("index_disposition"),
    col("readmit.encounter_id").alias("readmit_encounter_id"),
    col("readmit.encounter_date").alias("readmit_date"),
    when(col("readmit.encounter_id").isNotNull(), True).otherwise(False).alias("was_readmitted"),
    when(col("readmit.encounter_id").isNotNull(),
         datediff(col("readmit.encounter_date"), col("idx.discharge_date"))
    ).alias("days_to_readmission")
).dropDuplicates(["index_encounter_id"])  # Keep only first readmission per index

gold_readmissions.write.mode("overwrite").format("delta").saveAsTable("gold_readmissions")

# Validate: compare to national benchmark (~15%)
total = gold_readmissions.count()
readmitted = gold_readmissions.filter(col("was_readmitted") == True).count()
rate = (readmitted / total * 100) if total > 0 else 0
print(f"[x] gold_readmissions: {total} index admissions")
print(f"  Readmitted within 30 days: {readmitted} ({rate:.1f}%)")

print("\nReadmission Rate by Diagnosis:")
gold_readmissions.groupBy("index_diagnosis") \
    .agg(
        count("*").alias("total_admissions"),
        sum(when(col("was_readmitted"), 1).otherwise(0)).alias("readmissions"),
        round(sum(when(col("was_readmitted"), 1).otherwise(0)) / count("*") * 100, 1).alias("readmission_rate_pct")
    ) \
    .filter(col("total_admissions") >= 5) \
    .orderBy("readmission_rate_pct", ascending=False) \
    .show(15, truncate=False)

## ED Utilization & Frequent Flyers

### Business Context
**ED frequent flyers** are patients with 4+ ED visits per year. They represent a small fraction of the population but consume a disproportionate share of emergency resources:

- Top 5% of ED utilizers account for ~30% of all ED visits
- Each ED visit costs $2,000-$5,000 vs. $150-$300 for primary care
- Many frequent flyers have unmanaged chronic conditions that could be treated in outpatient settings at 1/10th the cost
- Identifying them enables **care management outreach** -- connecting patients with primary care, social workers, and community health resources

### Technical Approach
1. Filter to ED encounters, count visits per patient per year
2. Flag anyone with 4+ visits as a frequent flyer
3. Track `facilities_visited` -- visiting multiple EDs ("ED shopping") suggests coordination gaps
4. Join with patient demographics to profile the frequent flyer population

In [ ]:
encounters = spark.table("silver_encounters")
patients = spark.table("silver_patients")

# Count ED visits per patient per year
ed_visits = encounters \
    .filter(col("encounter_type") == "ED") \
    .groupBy("patient_id", "encounter_year") \
    .agg(
        count("*").alias("ed_visit_count"),
        countDistinct("facility_name").alias("facilities_visited"),
        sum("total_charges").alias("total_ed_charges")
    )

# Flag frequent flyers (4+ visits/year) and join with demographics
ed_frequent_flyers = ed_visits \
    .withColumn("is_frequent_flyer", when(col("ed_visit_count") >= 4, True).otherwise(False)) \
    .join(patients.select("patient_id", "first_name", "last_name", "age", "insurance_type", "risk_score"),
          "patient_id", "left")

ed_frequent_flyers.write.mode("overwrite").format("delta").saveAsTable("gold_ed_utilization")

total_ed_patients = ed_frequent_flyers.count()
frequent_flyers = ed_frequent_flyers.filter(col("is_frequent_flyer") == True).count()
print(f"[x] gold_ed_utilization: {total_ed_patients} patient-year records")
print(f"  Frequent flyers (4+ visits/year): {frequent_flyers}")

print("\nED Visit Distribution:")
ed_frequent_flyers.groupBy("ed_visit_count").count().orderBy("ed_visit_count").show()

## Average Length of Stay (ALOS) by Diagnosis

### Business Context
ALOS is a core **operational efficiency metric** that directly impacts cost, capacity, and quality:

- **Cost:** Each additional inpatient day costs $2,000-$3,000
- **Capacity:** Longer stays = fewer available beds = longer ED wait times ("boarding")
- **Quality signal:** ALOS outliers may indicate complications or discharge planning delays
- **National benchmark:** ~4.5 days for all inpatient stays

### Technical Approach
- Group by diagnosis AND facility to enable cross-facility benchmarking
- Compute `avg`, `stddev`, `min`, `max` -- high stddev means inconsistent care
- Filter to `LOS > 0` to exclude same-day observation stays miscoded as inpatient

In [ ]:
encounters = spark.table("silver_encounters")

# ALOS by diagnosis and facility (enables cross-facility benchmarking)
gold_alos = encounters \
    .filter((col("encounter_type") == "Inpatient") & (col("length_of_stay_days") > 0)) \
    .groupBy(
        "primary_diagnosis_code",
        "primary_diagnosis_description",
        "facility_name"
    ) \
    .agg(
        count("*").alias("admission_count"),
        round(avg("length_of_stay_days"), 1).alias("avg_los"),
        round(stddev("length_of_stay_days"), 1).alias("stddev_los"),
        min("length_of_stay_days").alias("min_los"),
        max("length_of_stay_days").alias("max_los"),
        round(avg("total_charges"), 2).alias("avg_charges")
    )

gold_alos.write.mode("overwrite").format("delta").saveAsTable("gold_alos")
print(f"[x] gold_alos: {gold_alos.count()} diagnosis-facility combinations")

print("\nTop 10 Diagnoses by Average LOS:")
gold_alos.groupBy("primary_diagnosis_description") \
    .agg(
        sum("admission_count").alias("total_admissions"),
        round(avg("avg_los"), 1).alias("overall_avg_los")
    ) \
    .filter(col("total_admissions") >= 3) \
    .orderBy("overall_avg_los", ascending=False) \
    .show(10, truncate=False)

## Encounter Summary (Volume & Demographics)

### Business Context
This is the **central fact table** for Power BI -- the single most important Gold table. It combines encounter details with patient demographics into one wide, denormalized table that supports every slice-and-dice question:

- "How many Medicare ED visits did Metro General have in Q3?"
- "What's the average LOS for high-risk inpatient patients aged 75+?"
- "Which facility has the highest weekend admission rate?"

### Technical Approach: Denormalization
Instead of making Power BI join 2 tables at query time (slow), we pre-join encounters + patients here. This creates a **star schema** fact table optimized for interactive BI:
- Encounter facts (dates, type, facility, charges)
- Time dimensions (month, quarter, day_of_week)
- Patient dimensions (age_group, gender, insurance, risk_category)

In [ ]:
encounters = spark.table("silver_encounters")
patients = spark.table("silver_patients")

# Denormalize: join encounters + patient demographics into one wide table
gold_encounter_summary = encounters \
    .join(patients.select("patient_id", "age", "age_group", "gender", 
                          "insurance_type", "risk_category", "race"),
          "patient_id", "left") \
    .select(
        "encounter_id", "patient_id", "encounter_date", "discharge_date",
        "encounter_type", "facility_name", "department",
        "primary_diagnosis_code", "primary_diagnosis_description",
        "attending_provider", "discharge_disposition",
        "length_of_stay_days", "total_charges",
        "encounter_month", "encounter_year", "encounter_quarter",
        "day_of_week", "is_weekend", "los_category",
        "age", "age_group", "gender", "insurance_type", "risk_category", "race"
    )

gold_encounter_summary.write.mode("overwrite").format("delta").saveAsTable("gold_encounter_summary")
print(f"[x] gold_encounter_summary: {gold_encounter_summary.count()} rows")

print("\nMonthly Encounter Volumes:")
gold_encounter_summary.groupBy("encounter_month") \
    .agg(
        count("*").alias("total_encounters"),
        sum(when(col("encounter_type") == "ED", 1).otherwise(0)).alias("ed_visits"),
        sum(when(col("encounter_type") == "Inpatient", 1).otherwise(0)).alias("inpatient"),
        sum(when(col("encounter_type") == "Outpatient", 1).otherwise(0)).alias("outpatient")
    ) \
    .orderBy("encounter_month") \
    .show(30)

## Financial Analysis -- Revenue, Denials, Collections

### Business Context
Healthcare revenue cycle management tracks the journey of a claim from service delivery to payment:

1. Patient receives care -> Hospital submits a claim to the insurance payer
2. Payer adjudicates: **pay**, **deny**, or **pend**
3. Denied claims must be appealed -- each appeal costs $25-$50 in admin labor
4. The industry loses ~**$262 billion** to denied claims annually

### Key Metrics
| Metric | Target | What It Tells You |
|---|---|---|
| Collection rate | >85% | Cents on the dollar actually collected |
| Denial rate | <10% | Percentage of claims rejected by payers |
| Days to payment | <45 | Cash flow velocity |

### Technical Approach
Join claims with encounter context (type, facility, diagnosis) so Power BI can answer: "What's the denial rate for ED visits at Metro General?" or "Which diagnosis has the lowest collection rate?"

In [ ]:
claims = spark.table("silver_claims")
encounters = spark.table("silver_encounters")

# Enrich claims with encounter context for dimensional analysis
gold_financial = claims \
    .join(
        encounters.select("encounter_id", "encounter_type", "facility_name",
                         "department", "primary_diagnosis_description",
                         "encounter_month", "encounter_year"),
        "encounter_id", "left"
    )

gold_financial.write.mode("overwrite").format("delta").saveAsTable("gold_financial")

print("[x] gold_financial created")
print("\nClaims Denial Analysis by Payer:")
gold_financial.groupBy("payer") \
    .agg(
        count("*").alias("total_claims"),
        sum(when(col("claim_status") == "Denied", 1).otherwise(0)).alias("denied_claims"),
        round(sum(when(col("claim_status") == "Denied", 1).otherwise(0)) / count("*") * 100, 1).alias("denial_rate_pct"),
        round(sum("claim_amount"), 2).alias("total_billed"),
        round(sum("paid_amount"), 2).alias("total_collected"),
        round(sum("paid_amount") / sum("claim_amount") * 100, 1).alias("collection_rate_pct")
    ) \
    .orderBy("denial_rate_pct", ascending=False) \
    .show(truncate=False)

## Population Health -- Chronic Disease Prevalence

### Business Context
**Population health management** identifies groups of patients with chronic conditions to deliver targeted interventions BEFORE they need acute care. This is the shift from **reactive** ("treat the sick") to **proactive** ("keep people healthy") healthcare.

Key facts:
- Patients with **3+ chronic conditions** account for **70% of all healthcare spending**
- Common high-cost clusters:
  - **Cardiometabolic triad:** Diabetes + Hypertension + CKD -> high risk for heart failure, stroke, dialysis
  - **Cardiopulmonary:** CHF + COPD -> each condition worsens the other
  - **Mental health comorbidity:** Depression + any chronic disease -> doubles costs, halves medication adherence

### Technical Approach
1. `collect_set()` aggregates all unique condition categories into an array per patient
2. `array_contains()` checks if a specific condition exists in that array -> creates boolean flags
3. `coalesce(value, 0)` handles NULLs for patients with no chronic conditions
4. Multimorbidity tiers: None, Moderate (1-2), High (3+)

In [ ]:
conditions = spark.table("silver_conditions")
patients = spark.table("silver_patients")

# Count chronic conditions per patient (collect all categories into an array)
patient_condition_count = conditions \
    .filter(col("condition_type") == "Chronic") \
    .groupBy("patient_id") \
    .agg(
        count("*").alias("chronic_condition_count"),
        collect_set("condition_category").alias("condition_list")
    )

# Join with patients and compute condition flags + multimorbidity tiers
gold_population_health = patients \
    .join(patient_condition_count, "patient_id", "left") \
    .withColumn("chronic_condition_count", 
        coalesce(col("chronic_condition_count"), lit(0))) \
    .withColumn("has_diabetes", 
        array_contains(col("condition_list"), "Diabetes")) \
    .withColumn("has_heart_failure",
        array_contains(col("condition_list"), "Heart Failure")) \
    .withColumn("has_copd",
        array_contains(col("condition_list"), "COPD")) \
    .withColumn("has_hypertension",
        array_contains(col("condition_list"), "Hypertension")) \
    .withColumn("has_ckd",
        array_contains(col("condition_list"), "Chronic Kidney Disease")) \
    .withColumn("multimorbidity",
        when(col("chronic_condition_count") >= 3, "High (3+)")
        .when(col("chronic_condition_count") >= 1, "Moderate (1-2)")
        .otherwise("None")) \
    .drop("condition_list")

gold_population_health.write.mode("overwrite").format("delta").saveAsTable("gold_population_health")

# Validate prevalence rates against national benchmarks
print(f"[x] gold_population_health: {gold_population_health.count()} rows")
print("\nChronic Condition Prevalence:")
total_patients = gold_population_health.count()
for cond in ["has_diabetes", "has_heart_failure", "has_copd", "has_hypertension", "has_ckd"]:
    cnt = gold_population_health.filter(col(cond) == True).count()
    print(f"  {cond.replace('has_', '').replace('_', ' ').title()}: {cnt} ({cnt/total_patients*100:.1f}%)")

print(f"\nMultimorbidity Distribution:")
gold_population_health.groupBy("multimorbidity").count().orderBy("multimorbidity").show()

print("\n[OK] All Gold layer tables created!")

---
## [OK] Module 2 Complete!

### What Was Created

**7 Silver Tables** (cleansed, typed, standardized):
- `silver_patients`, `silver_encounters`, `silver_conditions`, `silver_claims`, `silver_medications`, `silver_vitals`, `silver_clinical_notes`

**6 Gold Tables** (business-ready KPIs):
- `gold_readmissions`, `gold_ed_utilization`, `gold_alos`, `gold_encounter_summary`, `gold_financial`, `gold_population_health`

### Discussion Points

1. **Readmission Rate:** Your hospital's rate is around 15%. What would a 1% reduction mean in CMS penalty savings?
2. **ED Frequent Flyers:** These patients often have unmanaged chronic conditions. How could proactive outreach reduce ED burden?
3. **ALOS:** Sepsis patients stay much longer than average. What does this mean for bed capacity and staffing?
4. **Denial Rates:** Notice how rates differ by payer. What administrative costs does this create?

### Next Steps
-> **Module 3:** Create a Semantic Model and Power BI dashboard on top of these Gold tables